### Import

In [1]:
import gc
import os
import sys
import re
import math
import numpy as np
import pandas as pd
import datetime as dt
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt 

from multiprocessing import Pool
from multiprocessing import cpu_count

pd.set_option('display.max_columns', 500)

### General parameters

In [3]:
path_hirid      = 'PATH TO DATA/HiRID/'
path_observ_csv = 'PATH TO DATA/HiRID/raw_stage/observation_tables/'
path_pharma_csv = 'PATH TO DATA/HiRID/raw_stage/pharma_records/'
path_timeseries = '../Data/Output/'
path_extract_csv= '../Data/csvExtract/'

In [4]:
PID = 'patientid'
VARID = 'variableid'
VALUE = 'value'
DATETIME  = 'datetime'
ENTERTIME = 'entertime'

### Read general patient info

In [5]:
general_info = pd.read_csv(path_hirid + 'general_table.csv')
general_info['admissiontime'] = pd.to_datetime(general_info['admissiontime'])

### Extend general patient info

In [6]:
def generate_extended_general_table(path_observ_csv, df_general_table):
    
    observation_tables_path = path_observ_csv + 'parquet'

    df_obs_tables = pd.read_parquet(observation_tables_path,
                                    engine="pyarrow",
                                    columns=[PID, DATETIME, VARID, VALUE],
                                    filters=[(VARID, "in", [10000450, 10000400, 9990002, 9990004])])
    
    df_dropped = df_obs_tables.drop_duplicates()
    
    df_dropped = df_dropped.groupby([PID, VARID]).apply(lambda x: x.sort_values([DATETIME], 
                                                        ascending=True).head(1)).reset_index(drop=True)
    
    df_additional_cols = (pd.pivot_table(df_dropped, index=PID, columns=VARID, values=VALUE).
                          rename(columns={10000450: "height",
                                          10000400: "weight",
                                          9990002 : "APACHE II Group",
                                          9990004 : "APACHE IV Group"}))

    df_out = df_general_table.merge(df_additional_cols, how="left", left_on=PID, right_index=True).reset_index(drop=True)
    df_out[PID] = df_out[PID].astype('int32')
    
    col_order = ['patientid', 'admissiontime', 'sex', 'age', 'height', 'weight', 
                 'APACHE II Group', 'APACHE IV Group', 'discharge_status']
    
    df_out = df_out[col_order]
    
    return df_out

In [7]:
extend_general_info = generate_extended_general_table(path_observ_csv, general_info)
extend_general_info = extend_general_info.set_index(PID)

### Read Variable Reference

In [8]:
METAVAR_ID = 'metavariableid'
METAVAR_UNIT = 'metavariableunit'
VARREF_LOWERBOUND = 'lowerbound'
VARREF_UPPERBOUND = 'upperbound'

In [9]:
varref = pd.read_csv(path_hirid + 'varref.tsv', sep="\t", encoding='cp1252', index_col=0)
varref['mean'] = np.round(varref['mean'], 1)
varref['standard_deviation'] = np.round(varref['standard_deviation'], 1)

In [10]:
reference = pd.read_csv(path_hirid + 'refrence.csv', encoding='cp1252', index_col=0)
reference['metavariableid'] = varref.metavariableid

In [11]:
for idx, row in varref.iterrows():
    
    matching_row = reference[(reference['variablename'] == row['variablename']) & (reference['metavariableid'] == row['metavariableid'])]
    
    if not matching_row.empty:
        varref.at[idx, 'metavariablename'] = matching_row['metavariablename'].values[0]
        varref.at[idx, 'lowerbound'] = matching_row['lowerbound'].values[0]
        varref.at[idx, 'upperbound'] = matching_row['upperbound'].values[0]

### Clean variable reference for pharma and observation 

In [12]:
def clean_reference_table(varref):

    pharmaref = varref[varref["type"] == "pharma"].rename(columns={"variableid": "pharmaid"})
    
    STEPS_PER_HOURS = 60
    enum_ref = {'very short': int(STEPS_PER_HOURS / 12), 
                'short': 1 * STEPS_PER_HOURS, 
                '4h' :   4 * STEPS_PER_HOURS,
                '6h' :   6 * STEPS_PER_HOURS, 
                '12h':  12 * STEPS_PER_HOURS, 
                '24h':  24 * STEPS_PER_HOURS,
                '3d' :  72 * STEPS_PER_HOURS}
    
    pharmaref.loc[:, "pharmaactingperiod_min"] = pharmaref.pharmaactingperiod.apply(
                                                 lambda x: enum_ref[x] if type(x) == str else np.nan)
    check_func = lambda x: float(x) if type(x)==float or "/" not in x else float(x.split("/")[0])/float(x.split("/")[1])
    pharmaref["unitconversionfactor"] = pharmaref.unitconversionfactor.apply(check_func)
    
    observref = varref[varref["type"] != "pharma"].copy()
    observref.drop(observref.index[observref.variableid.isnull()], inplace=True)
    observref["variableid"] = observref.variableid.astype(int)
    observref.set_index("variableid", inplace=True)
    
    return observref, pharmaref

In [13]:
observref, pharmaref = clean_reference_table(varref)

### List of unique meta variable ids

In [14]:
lst_omid = np.sort(observref[METAVAR_ID].unique())
lst_pmid = np.sort(pharmaref[METAVAR_ID].unique())
lst_cumul_vid = observref[observref.variablename.apply(lambda x: "/c" in x.lower() or "cumul" in x.lower())].index.tolist()

### Make a directory for each patient

In [15]:
def makedirs_cohort(general_info, output_path):
    
    cohort_list = general_info.patientid.unique()
    nb_unit_stays = cohort_list.shape[0]
    
    for i, patient_id in enumerate(cohort_list):
        
        sys.stdout.write('\rStayID {0} of {1}...'.format(i+1, nb_unit_stays))
        dn = os.path.join(output_path, str(patient_id))

        try:
            os.makedirs(dn)  
        except:
            pass
        
    sys.stdout.write('DONE!\n')

In [ ]:
makedirs_cohort(general_info, path_timeseries)

### Read Index

In [17]:
observ_idx = pd.read_csv(path_observ_csv + 'observation_tables_index.csv')
pharma_idx = pd.read_csv(path_pharma_csv + 'pharma_records_index.csv')

### Read One Patients

In [18]:
def read_patient_obs_phr(patient_id):
    
    part_id_observ = observ_idx[observ_idx.patientid == patient_id].part.values[0]
    part_id_pharma = pharma_idx[pharma_idx.patientid == patient_id].part.values[0]

    patient_obs_df = pd.read_parquet(path_observ_csv + 'parquet/' + 'part-' + str(part_id_observ) + '.parquet', engine="pyarrow")
    patient_phr_df = pd.read_parquet(path_pharma_csv + 'parquet/' + 'part-' + str(part_id_observ) + '.parquet', engine="pyarrow")

    patient_obs_df = patient_obs_df[patient_obs_df.patientid == patient_id].copy()
    patient_phr_df = patient_phr_df[patient_phr_df.patientid == patient_id].copy()
    
    patient_obs_df['datetime']  = pd.to_datetime(patient_obs_df['datetime'])
    patient_obs_df['entertime'] = pd.to_datetime(patient_obs_df['entertime'])
    patient_phr_df['givenat']   = pd.to_datetime(patient_phr_df['givenat'])
    patient_phr_df['enteredentryat'] = pd.to_datetime(patient_phr_df['enteredentryat'])
    
    return patient_obs_df, patient_phr_df

### Process observation table

In [19]:
HR_VARID  = 200
SHORT_GAP = 5 / 60

In [20]:
def aggregate_cols(wide_observ, observref):
    
    metavar_varid_dict = {m: [f"v{vid}" for vid in vars] for (m, vars) in
                          observref.reset_index().groupby(METAVAR_ID)[VARID]}

    metavar_cols = {}
    
    for vmid, varid_cols in metavar_varid_dict.items():
        
        varid_cols_avail = list(set(varid_cols).intersection(set(wide_observ.columns)))

        if len(varid_cols_avail) > 1:
            c = wide_observ.loc[:, varid_cols_avail].median(axis=1, numeric_only=True, skipna=True)
        elif len(varid_cols_avail) == 1:
            c = wide_observ.loc[:, varid_cols_avail[0]]
        else:
            c = np.zeros(wide_observ.shape[0])
            c[:] = np.nan

        metavar_cols[f"vm{vmid}"] = c

    wide_observ_new = pd.DataFrame(metavar_cols, index=wide_observ.index)

    return wide_observ_new

In [21]:
def convert_cumul_value_to_rate(df, cumul_urine_id_lst, general_table):
    
    pid = df.iloc[0][PID]

    rec_adm_time = general_table.loc[pid].admissiontime

    if df[df[VARID] == HR_VARID][VALUE].notnull().sum() > 0:
        hr_first_meas_time = df.loc[df[df[VARID] == HR_VARID][VALUE].notnull().index[0], DATETIME]
        esti_adm_time = min(rec_adm_time, hr_first_meas_time)
    else:
        esti_adm_time = rec_adm_time

    df_urine = df[df[VARID].isin(cumul_urine_id_lst)]

    if len(df_urine) == 0:
        return df
    else:
        for vid in df_urine[VARID].unique():
            df_tmp = df_urine[df_urine[VARID] == vid]  

            index_pre_general_table = df_tmp.index[df_tmp[DATETIME] < esti_adm_time - np.timedelta64(15 * 60 + 30, "s")]
                                                                                                     
            if len(index_pre_general_table) == 0:
                pass
            elif len(index_pre_general_table) == 1:

                index_pre_general_table = df_tmp.index[df_tmp[DATETIME] < esti_adm_time]
                df.loc[index_pre_general_table[0], DATETIME] = esti_adm_time
            else:
                index_pre_general_table = df_tmp.index[df_tmp[DATETIME] < esti_adm_time]
                df.drop(index_pre_general_table[:-1], inplace=True)
                df.loc[index_pre_general_table[-1], DATETIME] = esti_adm_time

            df_tmp = df[df[VARID] == vid]
            if df_tmp.duplicated([DATETIME]).sum() == 0:
                pass
            else:
                df.drop(df_tmp.index[df_tmp.duplicated([DATETIME])], inplace=True)

            if (df[VARID] == vid).sum() < 2:
                df.drop(df.index[df[VARID] == vid], inplace=True)
                continue

            df_tmp = df[df[VARID] == vid]
            t_reset = df_tmp[(df_tmp[VALUE].diff() < 0) | (
                    df_tmp.index == df_tmp.index[0])][DATETIME]  
            for i in np.arange(1, len(t_reset)):
                tmp = df_tmp[df_tmp[DATETIME] >= t_reset.iloc[i]]
                if i < len(t_reset) - 1:
                    tmp = tmp[tmp[DATETIME] < t_reset.iloc[i + 1]]
                df.loc[tmp.index, VALUE] += df.loc[df_tmp.index[df_tmp[DATETIME] < t_reset.iloc[i]][-1], VALUE]

            df_tmp = df[df[VARID] == vid]
            tdiff = (df_tmp[DATETIME].diff().iloc[1:] / np.timedelta64(1,'h'))
            if (tdiff < SHORT_GAP).sum() > 0:
                df.drop(df_tmp.index[1:][tdiff.values < SHORT_GAP], inplace=True)

            if (df[VARID] == vid).sum() < 2:
                df.drop(df.index[df[VARID] == vid], inplace=True)
                continue

            df_tmp = df[df[VARID] == vid]
            vdiff = df_tmp[VALUE].diff()
            try:
                assert ((vdiff < 0).sum() == 0)
            except AssertionError:
                import ipdb
                ipdb.set_trace()
        gc.collect()

        for vid in df_urine[VARID].unique():
            df_tmp = df[df[VARID] == vid]
            if len(df_tmp) == 0:
                continue
            elif len(df_tmp) == 1:
                continue
            else:
                tdiff = (df_tmp[DATETIME].diff() / np.timedelta64(1,'h'))
                df.loc[df_tmp.index[1:], VALUE] = (df_tmp[VALUE].diff().iloc[1:] / tdiff.iloc[1:]).values
                df.loc[df_tmp.index[0], VALUE] = 0

        return df

In [22]:
def _merge_duplicate_measurements(tmp, stddev_dict):
    
    assert tmp[VALUE].isna().sum() == 0
    assert not tmp.empty

    val_std = tmp[VALUE].std()
    all_vals_equal = (len(tmp) == 1) | (val_std == 0)
    vid = tmp[VARID].iloc[0]

    ret = tmp.iloc[-1]

    if all_vals_equal:
        return ret
    
    elif vid in stddev_dict.keys() and val_std <= 0.1 * stddev_dict[vid]:
        ret[VALUE] = tmp[VALUE].mean()
        return ret

    return ret

In [23]:
def drop_duplicates_non_pharma(df, stddev_dict):
    
    duplicated_index = df.duplicated([DATETIME, VARID], keep=False)

    df_dup = df[duplicated_index]
    df_non_dup = df[~duplicated_index]

    if df_dup.empty:
        df_ret = df
        
    else:
        df_dedup = (df_dup.reset_index().groupby([DATETIME, VARID]).
                    apply(_merge_duplicate_measurements, stddev_dict=stddev_dict))

        df_dedup = df_dedup[~df_dedup[DATETIME].isna()].set_index('index')
        df_ret = pd.concat([df_dedup, df_non_dup]).sort_values([DATETIME, VARID])

    assert (df_ret.duplicated([DATETIME, VARID], keep=False).sum() == 0)

    return df_ret

In [24]:
def drop_out_of_range_values(df, observref):
    
    bound_cols = [VARREF_LOWERBOUND, VARREF_UPPERBOUND]
    
    df_with_bounds = df.merge(observref[bound_cols], left_on=VARID, right_index=True, how='inner')
    df_filtered = df_with_bounds.query(
        f'{VARREF_LOWERBOUND}.isnull() or {VARREF_UPPERBOUND}.isnull() or ({VARREF_LOWERBOUND} <= value and value <= {VARREF_UPPERBOUND})',
        engine='python')
    
    df_filtered = df_filtered.drop(columns=bound_cols)

    return df_filtered

In [25]:
def transform_obs_table_fn(observ: pd.DataFrame, lst_cumul_vid, observref, general_table):
    
    p_id = observ.patientid.values[0]  
    valid_variables = set(observref.index)

    observ = observ.loc[observ[VARID].isin(valid_variables)]
    observ = observ[~observ[VALUE].isna()].sort_values([VARID, DATETIME, ENTERTIME])

    observ = drop_out_of_range_values(observ, observref)

    stddev_dict = {v: std for (v, std) in observref["standard_deviation"].items()}
    observ = observ.drop([ENTERTIME], axis=1)
    observ = observ.drop_duplicates()
    observ = drop_duplicates_non_pharma(observ, stddev_dict)

    if observ[VARID].isin(lst_cumul_vid).sum() > 0:
        observ = convert_cumul_value_to_rate(observ, lst_cumul_vid, general_table)

    observ.loc[:, VARID] = observ.variableid.apply(lambda x: "v%d" % x)

    if observ.empty:
        return pd.DataFrame()

    wide_observ = (pd.pivot_table(observ, values=VALUE, columns=VARID, index=DATETIME).sort_index())

    wide_aggregated = aggregate_cols(wide_observ, observref)

    binary_vmids = ["vm%d" % x for x in observref[observref[METAVAR_UNIT].apply(lambda x: x == "Binary")][METAVAR_ID].values]
    
    for col in binary_vmids:
        wide_aggregated.loc[:, col] = wide_aggregated[col].apply(lambda x: x if np.isnan(x) else float(x != 0))
        
    wide_aggregated['patientid'] = p_id

    return wide_aggregated

### Process pharma table

In [26]:
PHARMAID = 'pharmaid'
PHARMA_DATETIME = 'givenat'
PHARMA_ENTERTIME = 'enteredentryat'
PHARMA_STATUS = 'recordstatus'
PHARMA_RATE = 'rate'
PHARMA_VAL = 'givendose'
INFID = 'infusionid'
UNITCONVERT_FACTOR = 'unitconversionfactor'
INVALID_PHARMA_STATUS  = [522, 526, 546, 782]
INSTANTANEOUS_STATE = 780
PSEUDO_INSTANTANEOUS_STATE = 544
START_STATE = 524
STOP_STATE  = 776

In [27]:
def process_instantaneous_state(df, acting_period):

    infusionid = int(df.iloc[0][INFID])

    df.set_index(PHARMA_DATETIME, inplace=True)
    drug_giventime_INSTANTANEOUS_STATE = df.index.tolist()

    df_new = []
    
    for i, dt in enumerate(drug_giventime_INSTANTANEOUS_STATE):
        tmp = df.loc[[dt]].copy()

        endtime_instantaneous_drug = dt + np.timedelta64(acting_period, "m")
        tmp.loc[endtime_instantaneous_drug, PHARMA_VAL] = tmp.loc[dt, PHARMA_VAL]
        tmp.loc[endtime_instantaneous_drug, PHARMA_STATUS] = STOP_STATE
        tmp.loc[endtime_instantaneous_drug, INFID] = "%d_%d" % (infusionid, i)

        tmp.loc[dt, PHARMA_VAL] = 0
        tmp.loc[dt, PHARMA_STATUS] = START_STATE
        tmp.loc[dt, INFID] = "%d_%d" % (infusionid, i)

        df_new.append(tmp.reset_index())
    df_new = pd.concat(df_new).sort_values(PHARMA_DATETIME)
    
    return df_new

In [28]:
def process_single_infusion(df, acting_period):

    infusionid = int(df.iloc[0][INFID])
    if len(df[PHARMA_STATUS].unique()) == 1 and df[PHARMA_STATUS].unique()[0] == INSTANTANEOUS_STATE:
        df = process_instantaneous_state(df, acting_period)

    df_rate = []
    
    for sub_infusionid in df[INFID].unique():
        tmp = df[df[INFID] == sub_infusionid].copy()
        try:
            assert ((tmp[PHARMA_STATUS] == START_STATE).sum() == 1)
        except AssertionError:
            tmp.set_index(PHARMA_DATETIME, inplace=True)
            beg_time = tmp.index[0] - np.timedelta64(acting_period, "m")
            tmp.loc[beg_time, PHARMA_VAL] = 0
            tmp.loc[beg_time, PHARMA_STATUS] = START_STATE
            tmp.loc[beg_time, INFID] = sub_infusionid
            tmp.sort_index(inplace=True)
            tmp.reset_index(inplace=True)
        try:
            assert ((tmp[PHARMA_STATUS] == STOP_STATE).sum() == 1)
        except AssertionError:
            pass
        tmp.loc[:, PHARMA_RATE] = 0
        tmp.loc[tmp.index[:-1], PHARMA_RATE] = tmp[PHARMA_VAL].values[1:] / (tmp[PHARMA_DATETIME].diff() / np.timedelta64(1,
                                                                                                          "m")).values[
                                                                     1:]
        tmp.rename(columns={PHARMA_RATE: str(sub_infusionid)}, inplace=True)
        df_rate.append(tmp[[PHARMA_DATETIME, str(sub_infusionid)]].set_index(PHARMA_DATETIME))
        
    df_rate = pd.concat(df_rate, axis=1).sum(axis=1).to_frame(name=str(infusionid))
    
    return df_rate

In [29]:
def drop_duplicates_pharma(df):

    df_dup = df[df.duplicated([PHARMA_DATETIME, PHARMAID, INFID], keep=False)]
    
    for pharmaid in df_dup[PHARMAID].unique():
        for infusionid in df_dup[df_dup[PHARMAID] == pharmaid][INFID].unique():
            tmp = df_dup[(df_dup[PHARMAID] == pharmaid) & (df_dup[INFID] == infusionid)]
            if len(tmp[PHARMA_STATUS].unique()) == 1 and tmp[PHARMA_STATUS].unique()[0] == INSTANTANEOUS_STATE:
                for i in range(len(tmp)):
                    df.loc[tmp.index[i], INFID] = "%s_%s" % (int(df.loc[tmp.index[i], INFID]), i)
            elif len(tmp[PHARMA_STATUS].unique()) == 1 and tmp[PHARMA_STATUS].unique()[0] == STOP_STATE:
                if (tmp[PHARMA_VAL] != 0).sum() == 1:
                    df.drop(tmp.index[tmp[PHARMA_VAL] == 0], inplace=True)
                else:
                    df.drop(tmp.index[:-1], inplace=True)
            elif len(tmp[PHARMA_STATUS].unique()) == 2 and STOP_STATE in tmp[PHARMA_STATUS].unique():
                df.drop(tmp.index[tmp[PHARMA_STATUS] != STOP_STATE], inplace=True)
            else:
                raise Exception("Debug needed")
                
    return df

In [30]:
def transform_pharma_table_fn(pharma: pd.DataFrame, pharmaref, lst_pmid):
        
    pharma_ids = set(pharmaref[PHARMAID].unique())
    pharma = pharma.loc[pharma[PHARMAID].isin(pharma_ids)].copy()

    pharma.drop(pharma.index[pharma[PHARMA_STATUS].isin(INVALID_PHARMA_STATUS)], inplace=True)
    pharma.sort_values([PHARMAID, PHARMA_DATETIME, PHARMA_ENTERTIME], inplace=True)
    pharma.loc[:, PHARMA_STATUS] = pharma[PHARMA_STATUS].replace(PSEUDO_INSTANTANEOUS_STATE, INSTANTANEOUS_STATE)
    pharma = drop_duplicates_pharma(pharma)

    if pharma.empty:
        return pd.DataFrame()
    else:
        wide_pharma = []
        for pharmaid in pharma[PHARMAID].unique():
            pharma_acting_period = pharmaref[pharmaref[PHARMAID] == pharmaid].iloc[0].pharmaactingperiod_min
            infusion_rate = []
            for infusionid in pharma[pharma[PHARMAID] == pharmaid][INFID].unique():
                tmp_pharma = pharma[(pharma[PHARMAID] == pharmaid) & (pharma[INFID] == infusionid)].copy()
                infusion_rate.append(process_single_infusion(tmp_pharma, pharma_acting_period))
            infusion_rate = pd.concat(infusion_rate, axis=1).sort_index()
            infusion_rate = infusion_rate.sum(axis=1).to_frame(name="p%d" % pharmaid)
            wide_pharma.append(infusion_rate)
            
        wide_pharma = pd.concat(wide_pharma, axis=1).sort_index()
        
        for pmid in lst_pmid:
            cols = ['p%d' % x for x in pharmaref[pharmaref[METAVAR_ID] == pmid][PHARMAID]]
            if np.isin(wide_pharma.columns, cols).sum() == 0:
                wide_pharma.loc[:, "pm%d" % pmid] = np.nan
            else:
                if pharmaref[pharmaref[METAVAR_ID] == pmid][UNITCONVERT_FACTOR].notnull().sum() > 0:
                    unitconverters = [pharmaref[(pharmaref[METAVAR_ID] == pmid) & (
                            pharmaref[PHARMAID] == int(c[1:]))].iloc[0][UNITCONVERT_FACTOR] for c in
                                      wide_pharma.columns[
                                          np.isin(wide_pharma.columns, cols)]]
                    wide_pharma.loc[:, "pm%d" % pmid] = (wide_pharma[
                                                             wide_pharma.columns[np.isin(wide_pharma.columns,
                                                                                         cols)]] * unitconverters).sum(
                        axis=1)
                else:
                    wide_pharma.loc[:, "pm%d" % pmid] = wide_pharma[
                        wide_pharma.columns[np.isin(wide_pharma.columns, cols)]].sum(axis=1)
                wide_pharma.loc[wide_pharma.index[
                                    wide_pharma[wide_pharma.columns[np.isin(wide_pharma.columns, cols)]].notnull().sum(
                                        axis=1) == 0], "pm%d" % pmid] = np.nan
                wide_pharma.drop(wide_pharma.columns[np.isin(wide_pharma.columns, cols)], axis=1, inplace=True)

        binary_pmids = ["pm%d" % x for x in
                        pharmaref[pharmaref[METAVAR_UNIT].apply(lambda x: x == "Binary")][METAVAR_ID].values]
        
        for col in binary_pmids:
            wide_pharma.loc[:, col] = wide_pharma[col].apply(lambda x: x if np.isnan(x) else float(x != 0))
        
        return wide_pharma

### Concatenate observation and pharma tables

In [31]:
output_cols = [PID, DATETIME] + [f"vm{vid}" for vid in sorted(observref[METAVAR_ID].unique())] + \
                                [f"pm{vid}" for vid in sorted(pharmaref[METAVAR_ID].unique())]

admission_times = {pid: adm_time for (pid, adm_time) in extend_general_info['admissiontime'].items()}

In [32]:
def combine_obs_and_pharma_tables(dfs, columns):
    
    assert len(dfs) == 2, "Expecting exactly two dictionaries"

    obs_df, ph_df = dfs
    p_id = obs_df.patientid[0]
    
    obs_df = obs_df.drop('patientid', axis=1)
    df_pid = pd.concat([obs_df, ph_df], axis=1)
    df_pid = df_pid.reset_index().rename(columns={"index": DATETIME, PHARMA_DATETIME: DATETIME})
    df_pid.insert(0, 'patientid', p_id)
    df_pid.loc[:, list(set(output_cols).difference(set(df_pid.columns)))] = np.nan
    df_pid = df_pid[output_cols]  

    assert ((df_pid.iloc[:, 2:].notnull().sum(axis=1) == 0).sum() == 0)

    df = df_pid.sort_values([PID, DATETIME])
    df[PID] = df[PID].astype('int32')
    df.iloc[:, 2:] = df.iloc[:, 2:].astype('float64')

    return df

### Filter on length of stay

In [33]:
HR_METAVAR_ID = 'vm1'

In [34]:
def length_of_stay_filtering(df, admission_time):
    
    rec_adm_time = admission_time
    
    if df[HR_METAVAR_ID].notnull().sum() > 0:
        hr_first_meas_time = df.loc[df[df[HR_METAVAR_ID].notnull()].index[0], DATETIME]
        esti_adm_time = max(rec_adm_time, hr_first_meas_time)
        esti_disc_time = df.loc[df[df[HR_METAVAR_ID].notnull()].index[-1], DATETIME]
    else:
        esti_adm_time = rec_adm_time
        esti_disc_time = None

    df = df.drop(df.index[df[DATETIME] < esti_adm_time])
    
    if esti_disc_time is not None:
        df = df.drop(df.index[df[DATETIME] > esti_disc_time])

    if not df.empty:
        los = (df.iloc[-1][DATETIME] - df.iloc[0][DATETIME]) / np.timedelta64(24, "h")
        los = np.round(los, 1)
        assert (los < 32)
        
    return df, los

### Binning Data

In [35]:
def rename_to_human_df(df, df_var_ref):
    
    to_rename = [k for k in df.columns if k[2:].isdigit()]
    meta_ids = [int(m[2:]) for m in to_rename]
    names = [str(df_var_ref[df_var_ref.metavariableid == i]['metavariablename'].values[0]) for i in meta_ids]
    final_names = [to_rename[i] if names[i] == 'nan' else names[i] for i in range(len(names))]
    rename_mapping = {old: new for old, new in zip(to_rename, final_names)}
    df = df.rename(columns=rename_mapping)
    
    return df

In [36]:
def resample_df(df, freq_string='5T'):
    
    cols = df.columns
    assert DATETIME in cols
    assert PID in cols

    def reorder_time(patient_sample):
        
        pid = patient_sample[PID].iloc[0]

        patient_sample = patient_sample.reset_index(drop=True)
        HRs_non_zero = np.where(~np.isnan(patient_sample.vm1))[0]
        
        if len(HRs_non_zero) > 0:

            HR_start_idx, HR_stop_idx = HRs_non_zero[0], HRs_non_zero[-1]
            patient_sample.loc[:HR_start_idx] = patient_sample.loc[:HR_start_idx].ffill()
        else:
            HR_start_idx, HR_stop_idx = 0, patient_sample.shape[0] - 1
            
        stay_stop_time, stay_start_time = patient_sample.loc[HR_stop_idx, DATETIME], patient_sample.loc[HR_start_idx, DATETIME]
        patient_sample = patient_sample.loc[HR_start_idx:HR_stop_idx].reset_index(drop=True)
        
        offset = np.timedelta64(stay_start_time.minute, 'm') + np.timedelta64(stay_start_time.second,'s') + np.timedelta64(
                 stay_start_time.microsecond, 'us') + np.timedelta64(1, 'us')
        
        patient_sample[DATETIME] = patient_sample[DATETIME] - offset

        grided = patient_sample.set_index(DATETIME).resample(freq_string, axis=0, closed='left', label='right').last().reset_index()
        grided.loc[:, DATETIME] -= (stay_start_time - offset + np.timedelta64(1, 'us'))

        grided = grided.reset_index(drop=True)
        grided[PID] = pid
        return grided

    dfs_pat = []
    for p in df[PID].unique():
        dfs_pat.append(reorder_time(df.query(f'{PID} == {p}')))
        gc.collect()

    df_part = pd.concat(dfs_pat).reset_index(drop=True)
    df_part[PID] = df_part[PID].astype('int64')
    df_part = df_part[[PID] + [c for c in df_part.columns if c != PID]]
    df_part[DATETIME] /= np.timedelta64(60, 's')
    
    return df_part

In [37]:
def irregular_to_gridded(df, df_var_ref, freq_string='5T'):
    
    df = resample_df(df, freq_string)
    gc.collect()
    df = rename_to_human_df(df, df_var_ref)
    
    return df

### Extract timeseries data based on ICU - multiprocessing

In [38]:
def process_csv(stay_dir):
    
    patient_id = int(stay_dir)
    dn = os.path.join(path_timeseries, stay_dir)
    
    try:
        sys.stdout.flush()
        
        patient_obs_df, patient_phr_df = read_patient_obs_phr(patient_id)
        
        patient_obs_df = transform_obs_table_fn(patient_obs_df, lst_cumul_vid, observref, extend_general_info)
        patient_phr_df = transform_pharma_table_fn(patient_phr_df, pharmaref, lst_pmid)
        patient_obs_phr_df = combine_obs_and_pharma_tables([patient_obs_df, patient_phr_df], output_cols)
        
        patient_obs_phr_df, los = length_of_stay_filtering(patient_obs_phr_df, admission_times[patient_id])
        
        patient_df_bin_05M = irregular_to_gridded(patient_obs_phr_df, varref, freq_string='5T')
        patient_df_bin_60M = irregular_to_gridded(patient_obs_phr_df, varref, freq_string='60T')
        
        patient_df_static = extend_general_info[extend_general_info.index == patient_id].copy()
        patient_df_static['Los'] = los
        patient_df_static = patient_df_static.reset_index()
        
        patient_df_static.to_csv(os.path.join( dn, 'admission.csv'), index=False)
        patient_df_bin_05M.to_csv(os.path.join(dn, 'raw_timeseries_05min.csv'), index=False)
        patient_df_bin_60M.to_csv(os.path.join(dn, 'raw_timeseries_60min.csv'), index=False)
        
    except Exception as e:
        print(f"Error processing {stay_dir}: {e}")
        exception_stayID.append(stay_dir)

In [39]:
dirs = os.listdir(path_timeseries)

exception_stayID = []
num_processes = cpu_count() 

In [ ]:
with Pool(num_processes) as p:
    for _ in tqdm(p.imap(process_csv, dirs), total=len(dirs)):
        pass